# Notebook 05 — Rigorous Evaluation and Production Artifacts

## RustWeatherML · PhD Team (Physics · Mathematics · Machine Learning)

### Protocol

1. **Refit** the tuned models (Nb04) on the full train set.
2. **Single prediction** on the test set — untouched by any prior notebook.
3. **Metrics with confidence intervals** via bootstrap ($B = 200$).
4. **Per-segment error analysis**: city, month, temperature bin.
5. **Classifier calibration**: Brier score and reliability curve.
6. **Skill score** relative to persistence-24 h.

### Multi-horizon models (NEW)

We train one Ridge regressor per forecast horizon:

- `temp_next_24h` → the original champion (`ridge_24h`)
- `temp_next_48h` → `ridge_48h`
- `temp_next_72h` → `ridge_72h`

All three share the same `alpha` (from Nb04) and the **same feature
ordering**, so the scaler is reused across models. We expect RMSE to
grow with horizon — weather dynamics decorrelate over time and the
model has less information about the distant future.

### Rain bagging ensemble (NEW)

The 30 `DecisionTreeClassifier` trees that Notebook 05 originally
trained only to plot the calibration curve are now persisted to disk.
At production time, the runtime code applies every tree, counts votes,
and divides by 30 to obtain a **calibrated rain probability** per
observation. The reliability curve from this same block is mapped into
a histogram calibrator so the raw vote-count probabilities can be
corrected further if they drift from the diagonal.

### Production-grade artifacts (updated)

| File | Purpose |
|---|---|
| `models/ridge_model.bin`       | 24 h Ridge (bincode) |
| `models/ridge_manual.json`     | 24 h coefficients + intercept (JSON) |
| `models/ridge_48h_model.bin`   | **48 h Ridge (bincode)** |
| `models/ridge_48h_manual.json` | **48 h coefficients + intercept** |
| `models/ridge_72h_model.bin`   | **72 h Ridge (bincode)** |
| `models/ridge_72h_manual.json` | **72 h coefficients + intercept** |
| `models/rain_rf_model.bin`     | RandomForestClassifier (single-call prediction) |
| `models/rain_bagging_ensemble.bin` | **30-tree DT bagging ensemble for probabilities** |
| `models/rain_calibration.json` | **10-bin reliability curve for probability calibration** |
| `models/scaler.json`           | means, stds, feature_names |
| `models/golden_test.json`      | 50 test-set rows with raw + z + all predictions |
| `models/production_contract.json` | expected metrics + SHA-256 of every binary |


In [1]:
:dep polars = { version = "0.46", features = ["lazy", "parquet"] }
:dep ndarray = { version = "0.16", features = ["serde", "approx"] }
:dep smartcore = { version = "0.3", features = ["serde"] }
:dep bincode = "1.3"
:dep rand = "0.8"
:dep sha2 = "0.10"
:dep serde = { version = "1.0", features = ["derive"] }
:dep serde_json = "1.0"

In [2]:
use polars::prelude::*;
use ndarray::{Array1, Array2, Axis};
use std::collections::{HashMap, BTreeMap};
use std::fs::File;
use std::io::{Read, Write};

use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::linalg::basic::arrays::Array;
use smartcore::linear::ridge_regression::{RidgeRegression, RidgeRegressionParameters};
use smartcore::linear::lasso::{Lasso, LassoParameters};
use smartcore::linear::logistic_regression::{LogisticRegression, LogisticRegressionParameters};
use smartcore::tree::decision_tree_regressor::{DecisionTreeRegressor, DecisionTreeRegressorParameters};
use smartcore::tree::decision_tree_classifier::{DecisionTreeClassifier, DecisionTreeClassifierParameters};
use smartcore::ensemble::random_forest_regressor::{RandomForestRegressor, RandomForestRegressorParameters};
use smartcore::ensemble::random_forest_classifier::{RandomForestClassifier, RandomForestClassifierParameters};

use rand::{rngs::StdRng, SeedableRng, Rng};
use sha2::{Sha256, Digest};

println!("Dependencies loaded.");

Dependencies loaded.


---
## 1. Load train and test data

In [3]:
let train_df = LazyFrame::scan_parquet("../data/features/train.parquet", Default::default())
    .unwrap().collect().unwrap();
let test_df = LazyFrame::scan_parquet("../data/features/test.parquet", Default::default())
    .unwrap().collect().unwrap();

let best_params_str = std::fs::read_to_string("../models/best_hyperparameters.json")
    .expect("Notebook 04 must have been executed first");
let best_params: serde_json::Value = serde_json::from_str(&best_params_str).unwrap();
let comparison_str = std::fs::read_to_string("../models/model_comparison.json").unwrap();
let comp: serde_json::Value = serde_json::from_str(&comparison_str).unwrap();
let final_features: Vec<String> = comp["feature_names"].as_array().unwrap()
    .iter().map(|v| v.as_str().unwrap().to_string()).collect();

println!("Train: {}  |  Test: {}  |  Features: {}",
         train_df.height(), test_df.height(), final_features.len());

Train: 20160  |  Test: 5712  |  Features: 80


In [4]:
fn df_to_array2(df: &DataFrame, cols: &[String]) -> Array2<f64> {
    let n_rows = df.height();
    let n_cols = cols.len();
    let mut data = Vec::with_capacity(n_rows * n_cols);
    for c in cols {
        let s = df.column(c.as_str()).unwrap();
        let f = s.cast(&DataType::Float64).unwrap();
        let ca = f.f64().unwrap().to_vec();
        for v in ca { data.push(v.unwrap_or(0.0)); }
    }
    Array2::from_shape_vec((n_cols, n_rows), data).unwrap().t().to_owned()
}

fn df_to_array1(df: &DataFrame, name: &str) -> Array1<f64> {
    let v: Vec<f64> = df.column(name).unwrap()
        .cast(&DataType::Float64).unwrap()
        .f64().unwrap().to_vec().into_iter()
        .map(|x| x.unwrap_or(0.0)).collect();
    Array1::from_vec(v)
}

fn ndarray_to_dense(a: &Array2<f64>) -> DenseMatrix<f64> {
    DenseMatrix::from_2d_vec(&a.outer_iter().map(|r| r.to_vec()).collect::<Vec<_>>())
}

println!("Helpers ready.");

Helpers ready.


In [5]:
// Strict filter: all three targets must be non-null so that every
// horizon trains on the same row set. The lag-48h guard is kept from
// the original Nb05.
let train_clean = train_df.clone().lazy().filter(
    col("temp_next_24h").is_not_null()
    .and(col("temp_next_48h").is_not_null())
    .and(col("temp_next_72h").is_not_null())
    .and(col("will_rain_next_24h").is_not_null())
    .and(col("temp_lag48h").is_not_null())
).collect().unwrap();

let test_clean = test_df.clone().lazy().filter(
    col("temp_next_24h").is_not_null()
    .and(col("temp_next_48h").is_not_null())
    .and(col("temp_next_72h").is_not_null())
    .and(col("will_rain_next_24h").is_not_null())
    .and(col("temp_lag48h").is_not_null())
).collect().unwrap();

println!("Train cleaned: {}", train_clean.height());
println!("Test  cleaned: {}", test_clean.height());

let X_train = df_to_array2(&train_clean, &final_features);
let X_test  = df_to_array2(&test_clean,  &final_features);

// Three horizons
let y_train_temp   = df_to_array1(&train_clean, "temp_next_24h");
let y_test_temp    = df_to_array1(&test_clean,  "temp_next_24h");
let y_train_temp48 = df_to_array1(&train_clean, "temp_next_48h");
let y_test_temp48  = df_to_array1(&test_clean,  "temp_next_48h");
let y_train_temp72 = df_to_array1(&train_clean, "temp_next_72h");
let y_test_temp72  = df_to_array1(&test_clean,  "temp_next_72h");

let y_train_rain: Vec<u32> = df_to_array1(&train_clean, "will_rain_next_24h").iter().map(|&v| v as u32).collect();
let y_test_rain:  Vec<u32> = df_to_array1(&test_clean,  "will_rain_next_24h").iter().map(|&v| v as u32).collect();

// Standardization
fn fit_scaler(x: &Array2<f64>) -> (Vec<f64>, Vec<f64>) {
    let n_feat = x.ncols();
    let mut means = vec![0.0_f64; n_feat];
    let mut stds  = vec![1.0_f64; n_feat];
    for j in 0..n_feat {
        let c = x.column(j);
        let mu = c.mean().unwrap_or(0.0);
        let var: f64 = c.iter().map(|v| (v - mu).powi(2)).sum::<f64>() / (c.len() as f64 - 1.0).max(1.0);
        means[j] = mu;
        stds[j]  = var.sqrt().max(1e-8);
    }
    (means, stds)
}

fn standardize(x: &Array2<f64>, means: &[f64], stds: &[f64]) -> Array2<f64> {
    let mut o = x.clone();
    for j in 0..x.ncols() {
        for i in 0..x.nrows() {
            o[[i, j]] = (x[[i, j]] - means[j]) / stds[j];
        }
    }
    o
}

let (means, stds) = fit_scaler(&X_train);
let X_train_z = standardize(&X_train, &means, &stds);
let X_test_z  = standardize(&X_test,  &means, &stds);

let X_train_dm   = ndarray_to_dense(&X_train);
let X_test_dm    = ndarray_to_dense(&X_test);
let X_train_dm_z = ndarray_to_dense(&X_train_z);
let X_test_dm_z  = ndarray_to_dense(&X_test_z);

println!("Matrices ready: X_train {:?}, X_test {:?}", X_train.shape(), X_test.shape());

Train cleaned: 19488


Test  cleaned: 4704


Matrices ready: X_train [19488, 80], X_test [4704, 80]


---
## 2. Metrics with bootstrap confidence intervals

In [6]:
fn rmse(yt: &[f64], yp: &[f64]) -> f64 {
    let n = yt.len() as f64;
    let s: f64 = yt.iter().zip(yp.iter()).map(|(a,b)| (a-b).powi(2)).sum();
    (s / n).sqrt()
}
fn mae(yt: &[f64], yp: &[f64]) -> f64 {
    yt.iter().zip(yp.iter()).map(|(a,b)| (a-b).abs()).sum::<f64>() / yt.len() as f64
}
fn mbe(yt: &[f64], yp: &[f64]) -> f64 {
    yp.iter().zip(yt.iter()).map(|(p,t)| p - t).sum::<f64>() / yt.len() as f64
}
fn r2(yt: &[f64], yp: &[f64]) -> f64 {
    let mean = yt.iter().sum::<f64>() / yt.len() as f64;
    let ss_res: f64 = yt.iter().zip(yp.iter()).map(|(t,p)| (t-p).powi(2)).sum();
    let ss_tot: f64 = yt.iter().map(|t| (t-mean).powi(2)).sum();
    if ss_tot > 1e-18 { 1.0 - ss_res / ss_tot } else { 0.0 }
}

fn bootstrap_rmse(yt: &[f64], yp: &[f64], b: usize, seed: u64) -> (f64, f64, f64) {
    let mut rng = StdRng::seed_from_u64(seed);
    let n = yt.len();
    let mut samples: Vec<f64> = Vec::with_capacity(b);
    for _ in 0..b {
        let idx: Vec<usize> = (0..n).map(|_| rng.gen_range(0..n)).collect();
        let yts: Vec<f64> = idx.iter().map(|&i| yt[i]).collect();
        let yps: Vec<f64> = idx.iter().map(|&i| yp[i]).collect();
        samples.push(rmse(&yts, &yps));
    }
    samples.sort_by(|a, b| a.partial_cmp(b).unwrap());
    let p2 = samples[(b as f64 * 0.025) as usize];
    let p50 = samples[b / 2];
    let p97 = samples[(b as f64 * 0.975) as usize];
    (p2, p50, p97)
}

fn mcc(yt: &[u32], yp: &[u32]) -> f64 {
    let mut tp = 0i64; let mut tn = 0i64; let mut fp = 0i64; let mut fnn = 0i64;
    for (t, p) in yt.iter().zip(yp.iter()) {
        match (*t, *p) {
            (1,1) => tp += 1, (0,0) => tn += 1,
            (0,1) => fp += 1, (1,0) => fnn += 1,
            _ => {}
        }
    }
    let denom = ((tp+fp) as f64 * (tp+fnn) as f64 * (tn+fp) as f64 * (tn+fnn) as f64).sqrt();
    if denom > 0.0 { (tp as f64 * tn as f64 - fp as f64 * fnn as f64) / denom } else { 0.0 }
}

fn cls_f1(yt: &[u32], yp: &[u32]) -> f64 {
    let mut tp = 0usize; let mut fp = 0usize; let mut fnn = 0usize;
    for (t, p) in yt.iter().zip(yp.iter()) {
        if *t == 1 && *p == 1 { tp += 1; }
        else if *t == 0 && *p == 1 { fp += 1; }
        else if *t == 1 && *p == 0 { fnn += 1; }
    }
    let prec = if tp + fp > 0 { tp as f64 / (tp + fp) as f64 } else { 0.0 };
    let rec  = if tp + fnn > 0 { tp as f64 / (tp + fnn) as f64 } else { 0.0 };
    if prec + rec > 0.0 { 2.0 * prec * rec / (prec + rec) } else { 0.0 }
}

println!("Metrics defined.");

Metrics defined.


---
## 3. Final training on the full train

We train three Ridge regressors (one per horizon) plus the RFC and the
supporting baselines. All regressors share the same alpha and feature
ordering.

In [7]:
let lasso_alpha: f64 = best_params["regression"]["lasso"]["alpha"].as_f64().unwrap_or(1.0);
let ridge_alpha: f64 = best_params["regression"]["ridge"]["alpha"].as_f64().unwrap_or(10.0);
let rf_trees:   usize = best_params["regression"]["random_forest"]["n_trees"].as_u64().unwrap_or(150) as usize;
let rf_depth:   u16   = best_params["regression"]["random_forest"]["max_depth"].as_u64().unwrap_or(18) as u16;
let gb_trees:   usize = best_params["regression"]["gradient_boosting"]["n_trees"].as_u64().unwrap_or(41) as usize;
let gb_depth:   u16   = best_params["regression"]["gradient_boosting"]["max_depth"].as_u64().unwrap_or(6) as u16;
let gb_eta:     f64   = best_params["regression"]["gradient_boosting"]["learning_rate"].as_f64().unwrap_or(0.09);
let rfc_trees:  usize = best_params["classification"]["random_forest"]["n_trees"].as_u64().unwrap_or(100) as usize;
let rfc_depth:  u16   = best_params["classification"]["random_forest"]["max_depth"].as_u64().unwrap_or(15) as u16;

let y_train_temp_v:   Vec<f64> = y_train_temp.to_vec();
let y_train_temp48_v: Vec<f64> = y_train_temp48.to_vec();
let y_train_temp72_v: Vec<f64> = y_train_temp72.to_vec();

println!("Training Lasso (alpha = {})...", lasso_alpha);
let lasso_model = Lasso::fit(&X_train_dm_z, &y_train_temp_v,
    LassoParameters::default().with_alpha(lasso_alpha)).unwrap();

println!("Training Ridge 24h (alpha = {})...", ridge_alpha);
let ridge_model = RidgeRegression::fit(&X_train_dm_z, &y_train_temp_v,
    RidgeRegressionParameters::default().with_alpha(ridge_alpha)).unwrap();

println!("Training Ridge 48h (alpha = {})...", ridge_alpha);
let ridge_model_48h = RidgeRegression::fit(&X_train_dm_z, &y_train_temp48_v,
    RidgeRegressionParameters::default().with_alpha(ridge_alpha)).unwrap();

println!("Training Ridge 72h (alpha = {})...", ridge_alpha);
let ridge_model_72h = RidgeRegression::fit(&X_train_dm_z, &y_train_temp72_v,
    RidgeRegressionParameters::default().with_alpha(ridge_alpha)).unwrap();

println!("Training RandomForestRegressor (n={}, d={})...", rf_trees, rf_depth);
let rf_model = RandomForestRegressor::fit(&X_train_dm, &y_train_temp_v,
    RandomForestRegressorParameters::default().with_n_trees(rf_trees).with_max_depth(rf_depth)).unwrap();

println!("Training GradientBoosting (K={}, d={}, eta={:.2})...", gb_trees, gb_depth, gb_eta);
let n_train_f = y_train_temp_v.len();
let n_test_f  = X_test.nrows();
let init_pred = y_train_temp_v.iter().sum::<f64>() / n_train_f as f64;
let mut gb_train_pred = vec![init_pred; n_train_f];
let mut gb_test_pred  = vec![init_pred; n_test_f];
for _ in 0..gb_trees {
    let res: Vec<f64> = y_train_temp_v.iter().zip(gb_train_pred.iter()).map(|(a,b)| a-b).collect();
    let tree = DecisionTreeRegressor::fit(&X_train_dm, &res,
        DecisionTreeRegressorParameters::default().with_max_depth(gb_depth)).unwrap();
    let upd_tr: Vec<f64> = tree.predict(&X_train_dm).unwrap();
    let upd_te: Vec<f64> = tree.predict(&X_test_dm).unwrap();
    for i in 0..n_train_f { gb_train_pred[i] += gb_eta * upd_tr[i]; }
    for i in 0..n_test_f  { gb_test_pred[i]  += gb_eta * upd_te[i]; }
}

println!("Training RandomForestClassifier (n={}, d={})...", rfc_trees, rfc_depth);
let rfc_model = RandomForestClassifier::fit(&X_train_dm, &y_train_rain,
    RandomForestClassifierParameters::default()
        .with_n_trees(rfc_trees as u16)
        .with_max_depth(rfc_depth)).unwrap();

println!("Training LogisticRegression...");
let log_model = LogisticRegression::fit(&X_train_dm_z, &y_train_rain, LogisticRegressionParameters::default()).unwrap();

println!("\nAll models trained (incl. 3 Ridge horizons).");

Training Lasso (alpha = 1)...


Training Ridge 24h (alpha = 10)...


Training Ridge 48h (alpha = 10)...


Training Ridge 72h (alpha = 10)...


Training RandomForestRegressor (n=150, d=18)...


Training GradientBoosting (K=41, d=6, eta=0.09)...


Training RandomForestClassifier (n=200, d=20)...


Training LogisticRegression...


All models trained (incl. 3 Ridge horizons).


In [8]:
// Predictions for every horizon.
let pred_lasso: Vec<f64> = lasso_model.predict(&X_test_dm_z).unwrap();
let pred_ridge:     Vec<f64> = ridge_model.predict(&X_test_dm_z).unwrap();
let pred_ridge_48h: Vec<f64> = ridge_model_48h.predict(&X_test_dm_z).unwrap();
let pred_ridge_72h: Vec<f64> = ridge_model_72h.predict(&X_test_dm_z).unwrap();
let pred_rf:        Vec<f64> = rf_model.predict(&X_test_dm).unwrap();
let pred_gb:        Vec<f64> = gb_test_pred.clone();

let pred_rfc: Vec<u32> = rfc_model.predict(&X_test_dm).unwrap();
let pred_log: Vec<u32> = log_model.predict(&X_test_dm_z).unwrap();

// Baselines
let pred_persist_24h = df_to_array1(&test_clean, "temp_lag24h").to_vec();
let positives: usize = y_train_rain.iter().filter(|&&v| v == 1).count();
let trivial_class: u32 = if 2*positives > y_train_rain.len() { 1 } else { 0 };
let pred_trivial: Vec<u32> = vec![trivial_class; y_test_rain.len()];

let pred_ensemble: Vec<f64> = pred_lasso.iter().zip(pred_gb.iter())
    .map(|(a, b)| 0.5*a + 0.5*b).collect();

println!("Predictions ready for {} test samples.", y_test_rain.len());

Predictions ready for 4704 test samples.


---
## 4. Metrics table with bootstrap 95% CI (all horizons)

In [9]:
let y_test_v   = y_test_temp.to_vec();
let y_test_48v = y_test_temp48.to_vec();
let y_test_72v = y_test_temp72.to_vec();

#[derive(Debug, Clone)]
struct RegRow {
    name: String,
    horizon: String,
    rmse: f64, mae: f64, r2: f64, mbe: f64,
    ci_lo: f64, ci_hi: f64,
}

fn eval_reg(name: &str, horizon: &str, yt: &[f64], yp: &[f64]) -> RegRow {
    let r = rmse(yt, yp);
    let (lo, _, hi) = bootstrap_rmse(yt, yp, 200, 42);
    RegRow {
        name: name.to_string(),
        horizon: horizon.to_string(),
        rmse: r,
        mae: mae(yt, yp),
        r2: r2(yt, yp),
        mbe: mbe(yt, yp),
        ci_lo: lo, ci_hi: hi,
    }
}

let rows = vec![
    eval_reg("Persistence-24h",     "24h", &y_test_v, &pred_persist_24h),
    eval_reg("Ridge (alpha=10)",    "24h", &y_test_v, &pred_ridge),
    eval_reg("Ridge 48h",           "48h", &y_test_48v, &pred_ridge_48h),
    eval_reg("Ridge 72h",           "72h", &y_test_72v, &pred_ridge_72h),
    eval_reg("Lasso (alpha=1.0)",   "24h", &y_test_v, &pred_lasso),
    eval_reg("RandomForest",        "24h", &y_test_v, &pred_rf),
    eval_reg("GradientBoosting",    "24h", &y_test_v, &pred_gb),
    eval_reg("Ensemble (Lasso+GB)", "24h", &y_test_v, &pred_ensemble),
];

let baseline_rmse = rows[0].rmse;

println!("=== TEST METRICS (regression) ===");
println!("{:<25} {:>6} {:>8} {:>12} {:>8} {:>8} {:>+8} {:>10}",
         "model", "horiz", "RMSE", "95% CI", "MAE", "R2", "MBE", "skill");
println!("{}", "-".repeat(92));
for r in &rows {
    let ss = if r.horizon == "24h" {
        1.0 - (r.rmse / baseline_rmse).powi(2)
    } else {
        0.0
    };
    println!("{:<25} {:>6} {:>8.3} [{:>4.2}, {:>4.2}] {:>8.3} {:>8.3} {:>+8.3} {:>10.4}",
             r.name, r.horizon, r.rmse, r.ci_lo, r.ci_hi, r.mae, r.r2, r.mbe, ss);
}

=== TEST METRICS (regression) ===


model                      horiz     RMSE       95% CI      MAE       R2      MBE      skill


--------------------------------------------------------------------------------------------


Persistence-24h              24h    4.077 [3.99, 4.18]    2.962    0.806   -1.099     0.0000


Ridge (alpha=10)             24h    3.529 [3.43, 3.66]    2.426    0.855   -0.690     0.2505


...

Lasso (alpha=1.0)            24h    3.581 [3.48, 3.70]    2.463    0.850   -0.799     0.2282


RandomForest                 24h    3.632 [3.52, 3.76]    2.461    0.846   -0.685     0.2064


GradientBoosting             24h    3.651 [3.54, 3.78]    2.502    0.844   -0.705     0.1977


Ensemble (Lasso+GB)          24h    3.587 [3.49, 3.72]    2.464    0.850   -0.752     0.2256


()

In [10]:
#[derive(Debug, Clone)]
struct ClsRow {
    name: String,
    acc: f64, f1: f64, mcc: f64,
    tn: usize, fp: usize, fn_: usize, tp: usize,
}

fn eval_cls(name: &str, yt: &[u32], yp: &[u32]) -> ClsRow {
    let n = yt.len() as f64;
    let acc = yt.iter().zip(yp.iter()).filter(|(t, p)| t == p).count() as f64 / n;
    let mut tp = 0usize; let mut tn = 0usize; let mut fp = 0usize; let mut fnn = 0usize;
    for (t, p) in yt.iter().zip(yp.iter()) {
        match (*t, *p) {
            (1,1) => tp += 1, (0,0) => tn += 1,
            (0,1) => fp += 1, (1,0) => fnn += 1,
            _ => {}
        }
    }
    ClsRow {
        name: name.to_string(),
        acc,
        f1: cls_f1(yt, yp),
        mcc: mcc(yt, yp),
        tn, fp, fn_: fnn, tp
    }
}

let cls_rows = vec![
    eval_cls("Trivial-majority",  &y_test_rain, &pred_trivial),
    eval_cls("LogisticRegression", &y_test_rain, &pred_log),
    eval_cls("RandomForest",       &y_test_rain, &pred_rfc),
];

println!("=== TEST METRICS (classification, will_rain_next_24h) ===");
println!("{:<22} {:>8} {:>8} {:>8}  {:>6} {:>6} {:>6} {:>6}",
         "model", "Acc", "F1", "MCC", "TN", "FP", "FN", "TP");
println!("{}", "-".repeat(76));
for r in &cls_rows {
    println!("{:<22} {:>8.3} {:>8.3} {:>8.3}  {:>6} {:>6} {:>6} {:>6}",
             r.name, r.acc, r.f1, r.mcc, r.tn, r.fp, r.fn_, r.tp);
}

=== TEST METRICS (classification, will_rain_next_24h) ===


model                       Acc       F1      MCC      TN     FP     FN     TP


----------------------------------------------------------------------------


Trivial-majority          0.710    0.830    0.000       0   1366      0   3338


LogisticRegression        0.845    0.895    0.608     870    496    232   3106


RandomForest              0.884    0.920    0.712    1023    343    203   3135


()

---
## 5. Error by city (24h winner)

In [11]:
// Winner of the 24h horizon = best non-baseline at index > 0
let mut winner_idx = 1usize;
for i in 2..rows.len() {
    if rows[i].horizon == "24h" && rows[i].rmse < rows[winner_idx].rmse {
        winner_idx = i;
    }
}
let winner_name = rows[winner_idx].name.clone();
println!("Winning 24h model: {}", winner_name);

let winner_pred: Vec<f64> = match winner_name.as_str() {
    s if s.starts_with("Lasso")           => pred_lasso.clone(),
    s if s.starts_with("Ridge (alpha")    => pred_ridge.clone(),
    s if s.starts_with("Ridge 48h")       => pred_ridge_48h.clone(),
    s if s.starts_with("Ridge 72h")       => pred_ridge_72h.clone(),
    s if s.starts_with("RandomForest")    => pred_rf.clone(),
    s if s.starts_with("GradientBoost")   => pred_gb.clone(),
    s if s.starts_with("Ensemble")        => pred_ensemble.clone(),
    _ => pred_ridge.clone(),
};

let cities_vec: Vec<String> = test_clean.column("city").unwrap().str().unwrap()
    .into_iter().map(|x| x.unwrap_or("").to_string()).collect();
let mut city_err: HashMap<String, Vec<f64>> = HashMap::new();
for i in 0..test_clean.height() {
    let err = y_test_temp[i] - winner_pred[i];
    city_err.entry(cities_vec[i].clone()).or_default().push(err);
}

let mut rows_city: Vec<(String, usize, f64, f64, f64)> = city_err.into_iter().map(|(c, v)| {
    let n = v.len();
    let rm: f64 = (v.iter().map(|e| e*e).sum::<f64>() / n as f64).sqrt();
    let mb: f64 = v.iter().sum::<f64>() / n as f64;
    let mx: f64 = v.iter().map(|e| e.abs()).fold(0.0_f64, f64::max);
    (c, n, rm, mb, mx)
}).collect();
rows_city.sort_by(|a, b| a.2.partial_cmp(&b.2).unwrap());

println!("\n=== RMSE PER CITY (winner: {}) ===", winner_name);
println!("{:<25} {:>8} {:>10} {:>+10} {:>10}", "city", "n", "RMSE", "MBE", "MaxErr");
for (c, n, r, mb, mx) in &rows_city {
    println!("{:<25} {:>8} {:>10.3} {:>+10.3} {:>10.3}", c, n, r, mb, mx);
}

Winning 24h model: Ridge (alpha=10)


=== RMSE PER CITY (winner: Ridge (alpha=10)) ===


city                             n       RMSE        MBE     MaxErr


Dubai                          336      1.997     +0.823      5.233


Los Angeles                    336      2.418     +0.394      8.233


...

Oslo                           336      4.106     +1.580     14.900


Shanghai                       336      4.205     +1.276     14.792


Chongqing                      336      4.444     +0.717     14.289


Nanjing                        336      5.167     +1.716     15.618


()

---
## 6. Rain classifier calibration — bagging ensemble with serialization

We build 30 `DecisionTreeClassifier` trees on bootstrap samples of the
training set, identical to the original Nb05 calibration block — but
now we **persist the trees** to disk as one bincode blob so the
production runtime can load them and compute calibrated probabilities.

In [12]:
let n_bag = 30usize;
let mut bag_rng = StdRng::seed_from_u64(1234);
let n_train_c = y_train_rain.len();

let mut bagging_trees: Vec<
    DecisionTreeClassifier<f64, u32, DenseMatrix<f64>, Vec<u32>>
> = Vec::with_capacity(n_bag);

println!("Training {} bagging trees for calibration + production proba...", n_bag);
for k in 0..n_bag {
    let idx: Vec<usize> = (0..n_train_c).map(|_| bag_rng.gen_range(0..n_train_c)).collect();
    let X_boot: Array2<f64> = {
        let n_cols = X_train.ncols();
        let mut data = Vec::with_capacity(idx.len() * n_cols);
        for &i in &idx {
            for j in 0..n_cols { data.push(X_train[[i, j]]); }
        }
        Array2::from_shape_vec((idx.len(), n_cols), data).unwrap()
    };
    let y_boot: Vec<u32> = idx.iter().map(|&i| y_train_rain[i]).collect();
    let X_boot_dm = ndarray_to_dense(&X_boot);
    let model = DecisionTreeClassifier::fit(&X_boot_dm, &y_boot,
        DecisionTreeClassifierParameters::default().with_max_depth(rfc_depth)).unwrap();
    bagging_trees.push(model);
    if (k + 1) % 10 == 0 {
        println!("  trained {} / {}", k + 1, n_bag);
    }
}

// Apply every tree to the test set and accumulate votes.
let mut vote_sum: Vec<usize> = vec![0; y_test_rain.len()];
for tree in &bagging_trees {
    let p: Vec<u32> = tree.predict(&X_test_dm).unwrap();
    for (i, v) in p.iter().enumerate() {
        if *v == 1 { vote_sum[i] += 1; }
    }
}

let proba: Vec<f64> = vote_sum.iter().map(|&v| v as f64 / n_bag as f64).collect();

// Brier score on raw proba
let n_test_c = y_test_rain.len() as f64;
let brier: f64 = proba.iter().zip(y_test_rain.iter())
    .map(|(p, y)| (p - (*y as f64)).powi(2)).sum::<f64>() / n_test_c;

let p_train = y_train_rain.iter().filter(|&&v| v == 1).count() as f64 / y_train_rain.len() as f64;
let brier_climo = p_train * (1.0 - p_train);
let brier_skill = 1.0 - brier / brier_climo;

println!("\n=== CLASSIFIER CALIBRATION ===");
println!("Bagging trees kept in memory: {}", bagging_trees.len());
println!("Brier score (raw proba):  {:.4}", brier);
println!("Brier climatology:        {:.4}", brier_climo);
println!("Brier skill score:        {:.4}  ({})", brier_skill,
    if brier_skill > 0.0 { "better than climatology" } else { "worse" });

Training 30 bagging trees for calibration + production proba...


  trained 10 / 30


  trained 20 / 30


  trained 30 / 30


=== CLASSIFIER CALIBRATION ===


Bagging trees kept in memory: 30


Brier score (raw proba):  0.0907


Brier climatology:        0.1938


Brier skill score:        0.5322  (better than climatology)


In [13]:
// 10-bin reliability curve.
let n_bins = 10usize;
let mut bin_counts = vec![0usize; n_bins];
let mut bin_pred_sum = vec![0.0_f64; n_bins];
let mut bin_obs_sum  = vec![0.0_f64; n_bins];
for (p, y) in proba.iter().zip(y_test_rain.iter()) {
    let b = ((p * n_bins as f64).floor() as usize).min(n_bins - 1);
    bin_counts[b] += 1;
    bin_pred_sum[b] += *p;
    bin_obs_sum[b]  += *y as f64;
}

println!("\n=== RELIABILITY CURVE (10 bins) ===");
println!("{:<12} {:>8} {:>12} {:>12} {:>10}",
         "bin", "n", "pred mean", "obs freq", "delta");
println!("{}", "-".repeat(58));

let mut calib_bins: Vec<serde_json::Value> = Vec::new();
for b in 0..n_bins {
    let n = bin_counts[b];
    let (pm, om) = if n > 0 {
        (bin_pred_sum[b] / n as f64, bin_obs_sum[b] / n as f64)
    } else {
        (b as f64 * 0.1 + 0.05, b as f64 * 0.1 + 0.05)  // midpoint fallback
    };
    if n >= 5 {
        println!("[{:.1},{:.1})  {:>8} {:>12.3} {:>12.3} {:>+10.3}",
                 b as f64 * 0.1, (b+1) as f64 * 0.1, n, pm, om, om - pm);
    }
    calib_bins.push(serde_json::json!({
        "bin_lo":   b as f64 * 0.1,
        "bin_hi":   (b + 1) as f64 * 0.1,
        "count":    n,
        "pred_mean": pm,
        "obs_freq":  om,
    }));
}

=== RELIABILITY CURVE (10 bins) ===


bin                 n    pred mean     obs freq      delta


----------------------------------------------------------


[0.0,0.1)       366        0.016        0.055     +0.038


[0.1,0.2)       258        0.128        0.120     -0.008


...

[0.6,0.7)       190        0.633        0.711     +0.077


[0.7,0.8)       222        0.737        0.653     -0.083


[0.8,0.9)       378        0.837        0.807     -0.031


[0.9,1.0)      2578        0.983        0.961     -0.022


()

---
## 7. Serialize every production artifact

Critical path for `src/`. We:

1. bincode-serialize the three Ridge models and the RFC.
2. Write the Ridge manual JSON (coefficients + intercept) for each horizon.
3. Write `scaler.json`.
4. bincode-serialize the full 30-tree bagging ensemble.
5. Write `rain_calibration.json` with the reliability curve.

In [14]:
std::fs::create_dir_all("../models").unwrap();

// --- 7.1 bincode Ridge models ---
let ridge_bin_path     = "../models/ridge_model.bin";
let ridge_48h_bin_path = "../models/ridge_48h_model.bin";
let ridge_72h_bin_path = "../models/ridge_72h_model.bin";

for (path, model) in [
    (ridge_bin_path,     &ridge_model),
    (ridge_48h_bin_path, &ridge_model_48h),
    (ridge_72h_bin_path, &ridge_model_72h),
] {
    let mut f = File::create(path).unwrap();
    let bytes = bincode::serialize(model).expect("bincode Ridge");
    f.write_all(&bytes).unwrap();
    println!("Saved {} ({} bytes)", path, bytes.len());
}

// --- 7.2 Ridge manual JSON (one per horizon) ---
fn extract_ridge_params(
    model: &RidgeRegression<f64, f64, DenseMatrix<f64>, Vec<f64>>,
) -> (Vec<f64>, (usize, usize), f64) {
    let coefs: &DenseMatrix<f64> = model.coefficients();
    let shape = coefs.shape();
    let mut v: Vec<f64> = Vec::with_capacity(shape.0 * shape.1);
    for i in 0..shape.0 {
        for j in 0..shape.1 {
            v.push(*coefs.get((i, j)));
        }
    }
    let intc: f64 = *model.intercept();
    (v, shape, intc)
}

let (coefs_vec, coefs_shape, intercept) = extract_ridge_params(&ridge_model);
let (coefs48_vec, coefs48_shape, intercept48) = extract_ridge_params(&ridge_model_48h);
let (coefs72_vec, coefs72_shape, intercept72) = extract_ridge_params(&ridge_model_72h);

fn ridge_manual_json(
    horizon: &str,
    alpha: f64,
    coefs: &[f64],
    shape: (usize, usize),
    intercept: f64,
    feature_names: &[String],
) -> serde_json::Value {
    serde_json::json!({
        "model_type": "ridge_regression",
        "horizon": horizon,
        "alpha": alpha,
        "intercept": intercept,
        "coefficients_shape": [shape.0, shape.1],
        "coefficients_row_major": coefs,
        "feature_names": feature_names,
        "feature_order_is_standardized": true,
        "formula": "y_hat = intercept + sum_j (coefs[j] * (x[j] - means[j]) / stds[j])",
    })
}

std::fs::write("../models/ridge_manual.json",
    serde_json::to_string_pretty(&ridge_manual_json("24h", ridge_alpha, &coefs_vec, coefs_shape, intercept, &final_features)).unwrap()
).unwrap();
std::fs::write("../models/ridge_48h_manual.json",
    serde_json::to_string_pretty(&ridge_manual_json("48h", ridge_alpha, &coefs48_vec, coefs48_shape, intercept48, &final_features)).unwrap()
).unwrap();
std::fs::write("../models/ridge_72h_manual.json",
    serde_json::to_string_pretty(&ridge_manual_json("72h", ridge_alpha, &coefs72_vec, coefs72_shape, intercept72, &final_features)).unwrap()
).unwrap();
println!("Saved ../models/ridge_{{24,48,72}}h_manual.json");

// --- 7.3 RFC bincode ---
let rfc_bin_path = "../models/rain_rf_model.bin";
{
    let mut f = File::create(rfc_bin_path).unwrap();
    let bytes = bincode::serialize(&rfc_model).expect("bincode RFC");
    f.write_all(&bytes).unwrap();
    println!("Saved {} ({} bytes)", rfc_bin_path, bytes.len());
}

// --- 7.4 Bagging ensemble bincode (Vec<DT>) ---
let bag_bin_path = "../models/rain_bagging_ensemble.bin";
{
    let mut f = File::create(bag_bin_path).unwrap();
    let bytes = bincode::serialize(&bagging_trees).expect("bincode bagging");
    f.write_all(&bytes).unwrap();
    println!("Saved {} ({} bytes, {} trees)", bag_bin_path, bytes.len(), bagging_trees.len());
}

// --- 7.5 Scaler JSON ---
let scaler_json = serde_json::json!({
    "version": "1.1.0",
    "n_features": final_features.len(),
    "feature_names": final_features,
    "means": means,
    "stds":  stds,
    "formula": "z[j] = (x[j] - means[j]) / stds[j]",
});
std::fs::write("../models/scaler.json",
    serde_json::to_string_pretty(&scaler_json).unwrap()).unwrap();
println!("Saved ../models/scaler.json");

// --- 7.6 Rain calibration JSON ---
let rain_calib_json = serde_json::json!({
    "version": "1.0.0",
    "source": "Notebook 05 bagging ensemble",
    "n_trees": bagging_trees.len(),
    "n_bins": n_bins,
    "reliability_bins": calib_bins,
    "brier_score":       brier,
    "brier_climatology": brier_climo,
    "brier_skill_score": brier_skill,
    "interpretation": "Given a vote count k / n_trees from the bagging ensemble, the bin containing k/n gives the expected observed frequency. Clip to [0,1].",
});
std::fs::write("../models/rain_calibration.json",
    serde_json::to_string_pretty(&rain_calib_json).unwrap()).unwrap();
println!("Saved ../models/rain_calibration.json");

Saved ../models/ridge_model.bin (675 bytes)


Saved ../models/ridge_48h_model.bin (675 bytes)


Saved ../models/ridge_72h_model.bin (675 bytes)


Saved ../models/ridge_{24,48,72}h_manual.json


Saved ../models/rain_rf_model.bin (8492595 bytes)


Saved ../models/rain_bagging_ensemble.bin (727090 bytes, 30 trees)


Saved ../models/scaler.json


Saved ../models/rain_calibration.json


---
## 8. Round-trip verification for all three Ridge horizons

In [15]:
// Load Ridge models back and assert bit-exact predictions.
for (label, path, expected) in [
    ("Ridge 24h", "../models/ridge_model.bin",     &pred_ridge),
    ("Ridge 48h", "../models/ridge_48h_model.bin", &pred_ridge_48h),
    ("Ridge 72h", "../models/ridge_72h_model.bin", &pred_ridge_72h),
] {
    let bytes = {
        let mut f = File::open(path).unwrap();
        let mut b = Vec::new();
        f.read_to_end(&mut b).unwrap();
        b
    };
    let loaded: RidgeRegression<f64, f64, DenseMatrix<f64>, Vec<f64>> =
        bincode::deserialize(&bytes).expect("deserialize Ridge");
    let pred_loaded: Vec<f64> = loaded.predict(&X_test_dm_z).unwrap();
    let max_diff: f64 = expected.iter().zip(pred_loaded.iter())
        .map(|(a, b)| (a - b).abs()).fold(0.0_f64, f64::max);
    println!("{} round-trip max diff: {:.2e}  (tol 1e-9)", label, max_diff);
    assert!(max_diff < 1e-9, "{} bincode serialization is not bit-exact!", label);
    println!("{} bincode round-trip verified bit-exact.", label);
}

Ridge 24h round-trip max diff: 0.00e0  (tol 1e-9)


Ridge 24h bincode round-trip verified bit-exact.


Ridge 48h round-trip max diff: 0.00e0  (tol 1e-9)


Ridge 48h bincode round-trip verified bit-exact.


Ridge 72h round-trip max diff: 0.00e0  (tol 1e-9)


Ridge 72h bincode round-trip verified bit-exact.


()

In [16]:
// RFC round-trip.
let rfc_bytes = {
    let mut f = File::open("../models/rain_rf_model.bin").unwrap();
    let mut b = Vec::new();
    f.read_to_end(&mut b).unwrap();
    b
};
let rfc_loaded: RandomForestClassifier<f64, u32, DenseMatrix<f64>, Vec<u32>> =
    bincode::deserialize(&rfc_bytes).expect("deserialize RFC");
let pred_rfc_loaded: Vec<u32> = rfc_loaded.predict(&X_test_dm).unwrap();
let n_match = pred_rfc.iter().zip(pred_rfc_loaded.iter()).filter(|(a, b)| a == b).count();
println!("RFC round-trip match: {} / {} ({:.2}%)",
    n_match, pred_rfc.len(), 100.0 * n_match as f64 / pred_rfc.len() as f64);
assert_eq!(pred_rfc, pred_rfc_loaded, "RFC bincode serialization is not bit-exact!");
println!("RFC round-trip verified bit-exact.");

// Bagging ensemble round-trip (re-vote and compare).
let bag_bytes = {
    let mut f = File::open("../models/rain_bagging_ensemble.bin").unwrap();
    let mut b = Vec::new();
    f.read_to_end(&mut b).unwrap();
    b
};
let bag_loaded: Vec<DecisionTreeClassifier<f64, u32, DenseMatrix<f64>, Vec<u32>>> =
    bincode::deserialize(&bag_bytes).expect("deserialize bagging ensemble");
let mut vote_sum2: Vec<usize> = vec![0; y_test_rain.len()];
for tree in &bag_loaded {
    let p: Vec<u32> = tree.predict(&X_test_dm).unwrap();
    for (i, v) in p.iter().enumerate() {
        if *v == 1 { vote_sum2[i] += 1; }
    }
}
let proba_loaded: Vec<f64> = vote_sum2.iter().map(|&v| v as f64 / bag_loaded.len() as f64).collect();
let max_proba_diff: f64 = proba.iter().zip(proba_loaded.iter())
    .map(|(a, b)| (a - b).abs()).fold(0.0_f64, f64::max);
println!("Bagging round-trip max proba diff: {:.2e}", max_proba_diff);
assert!(max_proba_diff < 1e-12, "Bagging ensemble lost predictions!");
println!("Bagging ensemble round-trip verified bit-exact ({} trees).", bag_loaded.len());

RFC round-trip match: 4704 / 4704 (100.00%)


RFC round-trip verified bit-exact.


Bagging round-trip max proba diff: 0.00e0


Bagging ensemble round-trip verified bit-exact (30 trees).


---
## 9. Golden test — 50 samples with raw + z + predictions at every horizon

In [17]:
let n_golden = 50usize;
let step = (test_clean.height() / n_golden).max(1);
let sample_idx: Vec<usize> = (0..test_clean.height()).step_by(step).take(n_golden).collect();

// Helper for calibration mapping
fn calibrate(raw: f64, bins: &[serde_json::Value]) -> f64 {
    let b_idx = ((raw * bins.len() as f64).floor() as usize).min(bins.len() - 1);
    let obs = bins[b_idx]["obs_freq"].as_f64().unwrap_or(raw);
    obs.max(0.0).min(1.0)
}

let mut golden_samples: Vec<serde_json::Value> = Vec::new();
for &i in &sample_idx {
    let raw_feats: Vec<f64> = (0..final_features.len()).map(|j| X_test[[i, j]]).collect();
    let z_feats:   Vec<f64> = (0..final_features.len()).map(|j| X_test_z[[i, j]]).collect();
    let city  = test_clean.column("city").unwrap().str().unwrap().get(i).unwrap_or("");
    let ts    = test_clean.column("timestamp").unwrap().str().unwrap().get(i).unwrap_or("");
    let rain_prob_raw = proba[i];
    let rain_prob_cal = calibrate(rain_prob_raw, &calib_bins);

    golden_samples.push(serde_json::json!({
        "row_index_in_test":  i,
        "city":               city,
        "timestamp":          ts,
        "raw_features":       raw_feats,
        "standardized_features": z_feats,
        "y_true_temp_next_24h": y_test_temp[i],
        "y_true_temp_next_48h": y_test_temp48[i],
        "y_true_temp_next_72h": y_test_temp72[i],
        "y_true_will_rain":     y_test_rain[i],
        "pred_ridge":           pred_ridge[i],
        "pred_ridge_48h":       pred_ridge_48h[i],
        "pred_ridge_72h":       pred_ridge_72h[i],
        "pred_lasso":           pred_lasso[i],
        "pred_rf":              pred_rf[i],
        "pred_gb":              pred_gb[i],
        "pred_rfc_class":       pred_rfc[i],
        "pred_rain_proba_raw":  rain_prob_raw,
        "pred_rain_proba_cal":  rain_prob_cal,
        "pred_log_class":       pred_log[i],
    }));
}

let golden = serde_json::json!({
    "version": "2.0.0",
    "generator": "Notebook 05",
    "tolerance_abs_regression": 1e-9,
    "tolerance_abs_classification": 0,
    "tolerance_abs_probability": 1e-9,
    "n_samples": sample_idx.len(),
    "feature_names": final_features,
    "samples": golden_samples,
});

std::fs::write("../models/golden_test.json",
    serde_json::to_string_pretty(&golden).unwrap()).unwrap();
println!("Saved ../models/golden_test.json ({} samples, v2.0 with 48/72h + calibrated proba)", sample_idx.len());

Saved ../models/golden_test.json (50 samples, v2.0 with 48/72h + calibrated proba)


---
## 10. Production contract with all artifacts

In [18]:
fn sha256_file(path: &str) -> String {
    let mut hasher = Sha256::new();
    let mut f = File::open(path).unwrap();
    let mut buf = Vec::new();
    f.read_to_end(&mut buf).unwrap();
    hasher.update(&buf);
    let result = hasher.finalize();
    format!("{:x}", result)
}

let ridge24_sha   = sha256_file("../models/ridge_model.bin");
let ridge48_sha   = sha256_file("../models/ridge_48h_model.bin");
let ridge72_sha   = sha256_file("../models/ridge_72h_model.bin");
let rfc_sha       = sha256_file("../models/rain_rf_model.bin");
let bagging_sha   = sha256_file("../models/rain_bagging_ensemble.bin");
let calib_sha     = sha256_file("../models/rain_calibration.json");
let scaler_sha    = sha256_file("../models/scaler.json");
let golden_sha    = sha256_file("../models/golden_test.json");

// Winner reg row and 48/72 rows
let winner_row: RegRow = rows.iter().find(|r| r.name == winner_name).unwrap().clone();
let row_48h: RegRow    = rows.iter().find(|r| r.horizon == "48h").unwrap().clone();
let row_72h: RegRow    = rows.iter().find(|r| r.horizon == "72h").unwrap().clone();

let cls_rf = cls_rows.iter().find(|r| r.name == "RandomForest").cloned();

let contract = serde_json::json!({
    "version": "2.0.0",
    "generator": "Notebook 05",
    "winner_model": winner_name,
    "expected_regression_metrics": {
        "rmse":  winner_row.rmse,
        "mae":   winner_row.mae,
        "r2":    winner_row.r2,
        "mbe":   winner_row.mbe,
        "rmse_95_ci": [winner_row.ci_lo, winner_row.ci_hi],
        "skill_vs_persistence_24h": 1.0 - (winner_row.rmse / baseline_rmse).powi(2),
    },
    "expected_regression_metrics_48h": {
        "rmse": row_48h.rmse, "mae": row_48h.mae, "r2": row_48h.r2, "mbe": row_48h.mbe,
        "rmse_95_ci": [row_48h.ci_lo, row_48h.ci_hi],
    },
    "expected_regression_metrics_72h": {
        "rmse": row_72h.rmse, "mae": row_72h.mae, "r2": row_72h.r2, "mbe": row_72h.mbe,
        "rmse_95_ci": [row_72h.ci_lo, row_72h.ci_hi],
    },
    "expected_classification_metrics": cls_rf.map(|r| serde_json::json!({
        "accuracy": r.acc, "f1": r.f1, "mcc": r.mcc,
        "confusion": { "tn": r.tn, "fp": r.fp, "fn": r.fn_, "tp": r.tp }
    })),
    "calibration": {
        "brier_score":       brier,
        "brier_climatology": brier_climo,
        "brier_skill_score": brier_skill,
    },
    "binary_hashes_sha256": {
        "ridge_model.bin":           ridge24_sha,
        "ridge_48h_model.bin":       ridge48_sha,
        "ridge_72h_model.bin":       ridge72_sha,
        "rain_rf_model.bin":         rfc_sha,
        "rain_bagging_ensemble.bin": bagging_sha,
        "rain_calibration.json":     calib_sha,
        "scaler.json":               scaler_sha,
        "golden_test.json":          golden_sha,
    },
    "tolerance_abs_regression":     1e-6,
    "tolerance_abs_golden_path":    1e-9,
    "tolerance_abs_probability":    1e-9,
    "n_features":                   final_features.len(),
    "feature_names":                final_features,
    "hyperparameters": {
        "ridge_alpha":       ridge_alpha,
        "lasso_alpha":       lasso_alpha,
        "rf_trees":          rf_trees,
        "rf_depth":          rf_depth,
        "gb_trees":          gb_trees,
        "gb_depth":          gb_depth,
        "gb_learning_rate":  gb_eta,
        "rfc_trees":         rfc_trees,
        "rfc_depth":         rfc_depth,
        "n_bagging_trees":   bagging_trees.len(),
    },
    "downstream_invariants": [
        "src/ must load all three Ridge models via bincode (or the manual JSON form)",
        "src/ must load rain_bagging_ensemble.bin for calibrated rain probabilities",
        "src/ must apply rain_calibration.json to map vote counts to observed frequencies",
        "src/ must use feature_names in this exact order",
        "tests/integration_loadmodel.rs must replay golden_test.json v2.0 within tolerance",
        "CI must assert binary_hashes_sha256 unchanged until a deliberate retrain"
    ]
});

std::fs::write("../models/production_contract.json",
    serde_json::to_string_pretty(&contract).unwrap()).unwrap();
println!("Saved ../models/production_contract.json (v2.0)");

println!("\nBinary hashes (first 16 chars):");
println!("  ridge_model.bin           {}", &ridge24_sha[..16]);
println!("  ridge_48h_model.bin       {}", &ridge48_sha[..16]);
println!("  ridge_72h_model.bin       {}", &ridge72_sha[..16]);
println!("  rain_rf_model.bin         {}", &rfc_sha[..16]);
println!("  rain_bagging_ensemble.bin {}", &bagging_sha[..16]);

Saved ../models/production_contract.json (v2.0)


Binary hashes (first 16 chars):


  ridge_model.bin           fc0b83029417911b


  ridge_48h_model.bin       1c4e40cefaa3f769


  ridge_72h_model.bin       0746c91ea0400f42


  rain_rf_model.bin         8876068022464269


  rain_bagging_ensemble.bin ce4165bb91cdaa16


---
## 11. Persist standard evaluation report

In [19]:
use serde_json::json;

let report = json!({
    "winner": winner_name,
    "regression": rows.iter().map(|r| json!({
        "model": r.name,
        "horizon": r.horizon,
        "rmse": r.rmse,
        "rmse_ci95": [r.ci_lo, r.ci_hi],
        "mae":  r.mae,
        "r2":   r.r2,
        "mbe":  r.mbe,
    })).collect::<Vec<_>>(),
    "classification": cls_rows.iter().map(|r| json!({
        "model": r.name,
        "accuracy": r.acc,
        "f1": r.f1,
        "mcc": r.mcc,
        "confusion": { "tn": r.tn, "fp": r.fp, "fn": r.fn_, "tp": r.tp }
    })).collect::<Vec<_>>(),
    "rmse_by_city": rows_city.iter().map(|(c, n, rm, mb, mx)| json!({
        "city": c, "n": n, "rmse": rm, "mbe": mb, "max_abs_err": mx
    })).collect::<Vec<_>>(),
    "calibration": {
        "brier_score":        brier,
        "brier_climatology":  brier_climo,
        "brier_skill_score":  brier_skill,
    },
});

std::fs::write("../models/evaluation_report.json",
    serde_json::to_string_pretty(&report).unwrap()).unwrap();
println!("Saved ../models/evaluation_report.json");

Saved ../models/evaluation_report.json


---
## 12. Final summary

In [20]:
println!("\n{}", "=".repeat(60));
println!("Notebook 05 complete (v2.0 with multi-horizon + bagging).");
println!("{}", "=".repeat(60));
println!("Winner 24h: {}", winner_name);
println!("Ridge 48h RMSE: {:.3}", row_48h.rmse);
println!("Ridge 72h RMSE: {:.3}", row_72h.rmse);
println!("Bagging trees serialized: {}", bagging_trees.len());
println!("Next: Notebook 06 - Drift Detection & Monitoring");

Notebook 05 complete (v2.0 with multi-horizon + bagging).


Winner 24h: Ridge (alpha=10)


Ridge 48h RMSE: 4.458


Ridge 72h RMSE: 5.065


Bagging trees serialized: 30


Next: Notebook 06 - Drift Detection & Monitoring
